# 위치 임베딩: 시퀀스 순서 인코딩 - 절대 위치 vs 상대 위치 임베딩

- Tutorial ID: `adv-2-1`
- Tutorial: 위치 임베딩: 시퀀스 순서 인코딩
- Section ID: `adv-2-1-1`
- Section: 절대 위치 vs 상대 위치 임베딩


In [ ]:
# ============================================================
# 코드 읽는 법 — 절대 위치 vs 상대 위치 임베딩
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 위치 정보가 벡터 회전/편향으로 attention score에 들어가는 방식 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

print("=" * 60)
print("위치 임베딩 (Positional Embeddings)")
print("=" * 60)

In [ ]:
# 1. 사인파 위치 임베딩

In [ ]:
print("\n1. 사인파 위치 임베딩 (Sinusoidal)")
print("-" * 45)

def sinusoidal_encoding(max_len, d_model):
    pe = np.zeros((max_len, d_model))
    position = np.arange(max_len)[:, np.newaxis]
    div_term = np.exp(np.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)
    return pe

pe = sinusoidal_encoding(128, 64)
print(f"PE shape: {pe.shape}")

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"\n위치 간 코사인 유사도:")
print(f"  pos 0 vs pos 1: {cosine_sim(pe[0], pe[1]):.4f}")
print(f"  pos 0 vs pos 5: {cosine_sim(pe[0], pe[5]):.4f}")
print(f"  pos 0 vs pos 50: {cosine_sim(pe[0], pe[50]):.4f}")
print(f"  → 가까운 위치일수록 유사!")

In [ ]:
# 2. RoPE (Rotary Position Embedding)

In [ ]:
print("\n2. RoPE (Rotary Position Embedding)")
print("-" * 45)

def apply_rope(x, pos, base=10000):
    d = len(x)
    theta = pos / (base ** (2 * np.arange(d // 2) / d))
    cos_t = np.cos(theta)
    sin_t = np.sin(theta)
    x_rot = np.zeros_like(x)
        for i in range(0, d, 2):
        x_rot[i] = x[i] * cos_t[i//2] - x[i+1] * sin_t[i//2]
        x_rot[i+1] = x[i] * sin_t[i//2] + x[i+1] * cos_t[i//2]
    return x_rot

d_head = 8
q = np.random.randn(d_head)
k = np.random.randn(d_head)

q0 = apply_rope(q, 0)
k0 = apply_rope(k, 0)
k5 = apply_rope(k, 5)

print(f"원래 q·k: {np.dot(q, k):.4f}")
print(f"같은 위치 q·k: {np.dot(q0, k0):.4f}")
print(f"다른 위치 q(0)·k(5): {np.dot(q0, k5):.4f}")
print("→ 같은 위치: 내적 보존, 다른 위치: 상대 거리에 따라 변화")

In [ ]:
# 3. ALiBi (Attention with Linear Biases)

In [ ]:
print("\n3. ALiBi")
print("-" * 45)

def alibi_bias(seq_len, num_heads):
    slopes = 2 ** (-8 * np.arange(1, num_heads + 1) / num_heads)
    positions = np.arange(seq_len)
    distances = positions[:, None] - positions[None, :]
    biases = []
        for m in slopes:
        bias = -m * np.abs(distances)
        bias = np.where(distances > 0, -1e9, bias)
        biases.append(bias)
    return np.stack(biases)

alibi = alibi_bias(6, 4)
print(f"ALiBi shape: {alibi.shape}")
print(f"헤드 0 (급격한 감소):\n{np.round(alibi[0], 2)}")

In [ ]:
# 4. 비교 요약

In [ ]:
print("\n4. 위치 임베딩 비교")
print("-" * 45)
print("""
| 방법       | 파라미터 | 외삽  | 상대위치 | 사용 모델     |
|------------|----------|-------|----------|---------------|
| Sinusoidal | 0        | 가능  | 아니오   | BERT, 원조    |
| Learned    | O(L×d)   | 불가  | 아니오   | GPT-2         |
| RoPE       | 0        | 가능  | 예       | LLaMA, PaLM   |
| ALiBi      | 0        | 가능  | 예       | BLOOM, MPT    |
""")